# ML Hyperparameter Tuning - Grid Search

This notebook demonstrates hyperparameter tuning for TempCNN using `ml_tune_grid`, which performs exhaustive grid search over all parameter combinations.

For random search (more efficient for large search spaces), see `05_ml_tuning_random.ipynb`.

After the search, the process **re-trains on the full sample set** with the best hyperparameters and returns that **trained model** (same idea as `ml_fit`). All evaluated runs remain in `tuning_results.json`. Chain **`save_ml_model`** on the tuning node to persist the model: if `tuning_results.json` exists in the job, it is added as an extra STAC asset (role `data`, not part of MLM).

## Runtime (why it can be slow)

Each **cell in the grid** runs a **full training** (all `epochs` you set) on the training sample set. Cost is roughly:

`#combinations × time_one_tempcnn_run`

So `batch_size` × `epochs` with 2×2 values = **4 full trainings** for scoring, **plus one final** full training on all samples with the best params. Deep models also **retry on CPU** if GPU memory is tight, which multiplies wall time.

**Ways to keep demos fast:** use **fewer / smaller grids**, **lower `epochs`**, `verbose: false`, keep **`cv: 0`** (single hold-out; k-fold multiplies work). For production tuning, prefer **`ml_tune_random`** with a modest `n_iter` and bounded `epochs` (see `05_ml_tuning_random.ipynb`).

In [1]:
import openeo # type: ignore

In [2]:
connection = openeo.connect(url="http://127.0.0.1:8000")
connection.authenticate_basic("brian", "123456")

<Connection to 'http://127.0.0.1:8000/' with BasicBearerAuth>

In [3]:
training_set = "https://github.com/e-sensing/sitsdata/raw/main/data/samples_deforestation_rondonia.rds"

In [4]:
tempcnn_model_init = connection.mlm_class_tempcnn(
    optimizer="adam",
    learning_rate=0.0005,
    seed=42,
)

In [5]:
# Default below is a small grid for shorter demos (2 combinations).
# For a serious search, widen epochs / batch_size or add commented axes — each
# new value multiplies total training runs.

param_grid = {
    "learning_rate": [0.0005, 0.0001],
    #"batch_size": [32, 64],
    "epochs": [20, 40],  # e.g. [50, 100] is much slower (4+ min per combo is common)
}

In [6]:
process_graph = {
    "tempcnn1": {
        "process_id": "mlm_class_tempcnn",
        "arguments": {
            "optimizer": "adam",
            "learning_rate": 0.0005,
            "seed": 42,
            "verbose": True
        },
    },
    "tune1": {
        "process_id": "ml_tune_grid",
        "arguments": {
            "model": {"from_node": "tempcnn1"},
            "training_data": training_set,
            "target": "label",
            "parameters": param_grid,
            "scoring": "accuracy",
            "cv": 0,
            "seed": 42
        },
    },
    "save1": {
        "process_id": "save_ml_model",
        "arguments": {
            "data": {"from_node": "tune1"},
            "name": "tempcnn-tuned-grid",
        },
        "result": True,
    },
}

job = connection.create_job(
    process_graph=process_graph,
    title="TempCNN hyperparameter tuning (grid search)",
    description="Exhaustive grid search for TempCNN hyperparameters",
)

In [7]:
job

<BatchJob job_id='111ae7f374c4383ebe8c6c5f2d89e9eb'>

In [8]:
job.start_and_wait()
results = job.get_results()

0:00:00 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': send 'start'
0:00:03 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:08 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:14 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:22 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:32 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:44 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:00:59 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:01:18 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:01:42 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:02:12 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:02:49 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:03:36 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:04:34 Job '111ae7f374c4383ebe8c6c5f2d89e9eb': running (progress N/A)
0:05:34 Job '111

In [9]:
results.download_files("data/outputs_grid_save_best/")

[PosixPath('data/outputs_grid_save_best/tuning_results'),
 PosixPath('data/outputs_grid_save_best/job-results.json')]